Part A - Write a function generate_housing_data(N, true_w, noise_std, seed) that:

Creates X of shape (N, len(true_w)) with random feature values (e.g. np.random.randn)
Computes y = X @ true_w + noise, where noise is drawn from a normal distribution with std noise_std
Returns X, y

Generate a dataset with N=200, true_w = np.array([3.0, -1.5]) (representing something like "price per sqft" and "price penalty per mile from city center"), noise_std=1.0.





Part B (Train/Test Split) - Manually slice (no sklearn train_test_split yet): first 160 rows train, last 40 test.




Part C (Fit and Compare) - Fit linear_regression_closed_form on training set only. Compare learned weights to true_w.




Part D (Evaluate Properly) - Compute train MSE and test MSE separately. Compare. Then retrain on all 200 points, compare training MSE to the original 160-point training MSE.




Part E (Bag-of-Words) - Given a small labeled reviews/labels dataset (6 reviews, positive/negative), build vocab, featurize with bag_of_words, fit closed-form regression, predict + round to 0/1, compute accuracy.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
#Part A - return x and y
#while y is unkown, problem provides true_w, as well as guidelines for generating x and noise, meaning that y can be derived from the equation y = (x*true_W) + noise
def generate_housing_data(N, true_w, noise_std, seed):
##np.random.seed(seed) guarantees that X and noise's first randomly generated output is reproduced for subsequent random draws(maintains consistency when the function gets called repeatedly)
  np.random.seed(seed)
  X = np.random.randn(N, len(true_w))
##draw N independent samples from a normal distribution with mean = 0, standard deviation = noise_std
  noise = np.random.randn(N)*noise_std
  Y = X @ true_w +noise
  return X, Y
X, Y = generate_housing_data(200, np.array([3.0,-1.5]), 1.0, 42)
#Part B - split the data into train and test
X_train = X[0:160, : ]
X_test = X[160:200, : ]
Y_train = Y[0:160]
Y_test = Y[160:200]
#Part C - compare closed-form solution for w using training data to true_w
def linear_regression_closed_form(X, Y):
  return np.linalg.solve(X.T @ X, X.T @ Y)
learned_w = linear_regression_closed_form(X_train, Y_train)
print(f"true_w = [3.0, -1.5], learned_w = {learned_w}")
#Part D - compare MSE of train, test, and full data
Y_train_predict = X_train @ learned_w
Y_test_predict = X_test @ learned_w
train_MSE = np.mean((Y_train - Y_train_predict)**2)
test_MSE = np.mean((Y_test - Y_test_predict)**2)
learned_w_full = linear_regression_closed_form(X, Y)
Y_total_predict = X @ learned_w_full
total_MSE = np.mean((Y - Y_total_predict)**2)
print(f"train_MSE = {train_MSE:.4f}, test_MSE = {test_MSE:.4f}, total_MSE = {total_MSE:.4f}")
#Part E - implement bag-of-words and compute its accuracy
reviews = [
    "great product loved it",
    "terrible waste of money",
    "excellent quality highly recommend",
    "awful do not buy",
    "amazing value great price",
    "poor quality very disappointed"
]
vocab = set()
for review in reviews:
  for word in review.lower().split():
    vocab.add(word)
def bag_of_words(reviews, vocab):
  sentences = []
  sentence_vocab_count = []
  for review in reviews:
    count = {word:0 for word in vocab}
    for word in review.lower().split():
      if word in vocab:
        count[word] += 1
  #sentence_vocab_count inputs the vocab count accumulated for that singular review, while sentences.append adds the row just built to a matrix that in the end will hold the vocab count for each review
    sentence_vocab_count = [count[word] for word in vocab]
    sentences.append(sentence_vocab_count)
  return np.array(sentences, dtype = float)
vocab = sorted(vocab)
labels = np.array([1, 0, 1, 0, 1, 0])
X_bow = bag_of_words(reviews, vocab)
model = LinearRegression()
model.fit(X_bow, labels)
predictions = model.predict(X_bow)
predictions_rounded = np.round(predictions)
prediction_accuracy = np.mean((labels==predictions_rounded))
print(f"bow_accuracy = {prediction_accuracy}")

true_w = [3.0, -1.5], learned_w = [ 3.09745077 -1.4647255 ]
train_MSE = 0.9789, test_MSE = 1.0147, total_MSE = 0.9850
bow_accuracy = 1.0
